# Fama‑French & Momentum — Clean and Merge

This notebook replaces the previous implementation with a robust, well‑documented pipeline that:

- Cleans the FF5 and Momentum CSVs
- Standardizes the date column and numeric formats
- Merges into a monthly combined dataset (drops months missing any factor)
- Aggregates to quarterly by averaging
- Saves cleaned monthly and quarterly CSVs for downstream use


In [6]:
# Imports and file paths
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

DATA_DIR = Path('.')
FF5_CSV = DATA_DIR / 'F-F_Research_Data_5_Factors_2x3.csv'
MOM_CSV = DATA_DIR / 'F-F_Momentum_Factor.csv'
OUT_MONTHLY = DATA_DIR / 'factor_returns_monthly_cleaned.csv'
OUT_QUARTERLY = DATA_DIR / 'factor_returns_quarterly_cleaned.csv'
OUT_FF5 = DATA_DIR / 'ff5_cleaned.csv'
OUT_MOM = DATA_DIR / 'mom_cleaned.csv'


In [7]:
def parse_yyyymm(col):
    return pd.to_datetime(col.astype(str), format='%Y%m', errors='coerce')


def clean_ff5(path=FF5_CSV):
    """Load and clean FF5 CSV, return DataFrame with DATE, Value, Quality."""
    df = pd.read_csv(path, skiprows=2, dtype=str)
    df.columns = df.columns.str.strip()
    # first column is the date
    date_col = df.columns[0]
    df.rename(columns={date_col: 'DATE'}, inplace=True)

    df['DATE'] = parse_yyyymm(df['DATE'])
    df = df.dropna(subset=['DATE']).copy()

    # numeric columns
    factor_cols = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']
    for c in factor_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # convert percent to decimal
    df[factor_cols] = df[factor_cols] / 100.0

    # select and rename
    df = df[['DATE', 'HML', 'RMW']].rename(columns={'HML': 'Value', 'RMW': 'Quality'})

    df = df.drop_duplicates(subset=['DATE']).sort_values('DATE').reset_index(drop=True)
    return df


def clean_momentum(path=MOM_CSV):
    """Load and clean Momentum CSV, return DataFrame with DATE, Momentum."""
    df = pd.read_csv(path, dtype=str, skiprows=13)
    df.columns = df.columns.str.strip()
    date_col = df.columns[0]
    df.rename(columns={date_col: 'DATE'}, inplace=True)

    df['DATE'] = parse_yyyymm(df['DATE'])
    df = df.dropna(subset=['DATE']).copy()

    # detect momentum column
    candidate_cols = [c for c in df.columns if 'mom' in c.lower() or 'wml' in c.lower() or 'momentum' in c.lower()]
    mom_col = candidate_cols[0] if candidate_cols else df.columns[1]

    df[mom_col] = pd.to_numeric(df[mom_col], errors='coerce')
    df[mom_col] = df[mom_col] / 100.0

    df = df[['DATE', mom_col]].rename(columns={mom_col: 'Momentum'})
    df = df.drop_duplicates(subset=['DATE']).sort_values('DATE').reset_index(drop=True)
    return df


def merge_monthly(ff5_df, mom_df, how='inner'):
    merged = ff5_df.merge(mom_df, on='DATE', how=how)
    # drop rows where any factor is missing
    merged = merged.dropna(subset=['Value', 'Quality', 'Momentum']).reset_index(drop=True)
    return merged


def aggregate_to_quarterly(df):
    q = df.copy()
    q['Quarter'] = q['DATE'].dt.to_period('Q')
    q = q.groupby('Quarter')[['Value', 'Quality', 'Momentum']].mean().reset_index()
    q['DATE'] = q['Quarter'].dt.to_timestamp()
    q = q.drop(columns='Quarter').sort_values('DATE').reset_index(drop=True)
    return q


# Run the pipeline
ff5_clean = clean_ff5()
mom_clean = clean_momentum()

print(f"✓ FF5: {len(ff5_clean)} months (from {ff5_clean['DATE'].min()} to {ff5_clean['DATE'].max()})")
print(f"✓ Momentum: {len(mom_clean)} months (from {mom_clean['DATE'].min()} to {mom_clean['DATE'].max()})")

factors_monthly = merge_monthly(ff5_clean, mom_clean, how='inner')
print(f"✓ Merged monthly dataset: {len(factors_monthly)} months (from {factors_monthly['DATE'].min()} to {factors_monthly['DATE'].max()})")

# Quality checks
print('\nSummary statistics:')
print(factors_monthly[['Value', 'Quality', 'Momentum']].describe())
print('\nMissing values (monthly):')
print(factors_monthly.isnull().sum())

# Save cleaned inputs and outputs
ff5_clean.to_csv(OUT_FF5, index=False)
mom_clean.to_csv(OUT_MOM, index=False)
factors_monthly.to_csv(OUT_MONTHLY, index=False)

factors_quarterly = aggregate_to_quarterly(factors_monthly)
factors_quarterly.to_csv(OUT_QUARTERLY, index=False)

print(f"\nSaved: {OUT_FF5.name}, {OUT_MOM.name}, {OUT_MONTHLY.name}, {OUT_QUARTERLY.name}")


✓ FF5: 738 months (from 1963-07-01 00:00:00 to 2024-12-01 00:00:00)
✓ Momentum: 1186 months (from 1927-01-01 00:00:00 to 2025-10-01 00:00:00)
✓ Merged monthly dataset: 738 months (from 1963-07-01 00:00:00 to 2024-12-01 00:00:00)

Summary statistics:
            Value     Quality    Momentum
count  738.000000  738.000000  738.000000
mean     0.002775    0.002849    0.006022
std      0.029948    0.022163    0.041911
min     -0.138800   -0.186500   -0.343400
25%     -0.014150   -0.008025   -0.009775
50%      0.001950    0.002550    0.006850
75%      0.017300    0.013175    0.028550
max      0.128000    0.130700    0.180200

Missing values (monthly):
DATE        0
Value       0
Quality     0
Momentum    0
dtype: int64

Saved: ff5_cleaned.csv, mom_cleaned.csv, factor_returns_monthly_cleaned.csv, factor_returns_quarterly_cleaned.csv


In [5]:
# Merge from saved cleaned CSVs and save result
OUT_MERGED_FROM_CLEANED = DATA_DIR / 'factors_merged_from_cleaned_inputs.csv'

ff5_from_file = pd.read_csv(OUT_FF5, parse_dates=['DATE'])
mom_from_file = pd.read_csv(OUT_MOM, parse_dates=['DATE'])

merged_from_files = ff5_from_file.merge(mom_from_file, on='DATE', how='inner')
merged_from_files = merged_from_files.dropna(subset=['Value', 'Quality', 'Momentum']).sort_values('DATE').reset_index(drop=True)

merged_from_files.to_csv(OUT_MERGED_FROM_CLEANED, index=False)
print(f"✓ Merged from cleaned CSVs saved: {OUT_MERGED_FROM_CLEANED.name} ({len(merged_from_files)} rows; {merged_from_files['DATE'].min()} to {merged_from_files['DATE'].max()})")
merged_from_files.head()


✓ Merged from cleaned CSVs saved: factors_merged_from_cleaned_inputs.csv (738 rows; 1963-07-01 00:00:00 to 2024-12-01 00:00:00)


,DATE,Value,Quality,Momentum
0,1963-07-01,-0.0097,0.0068,0.0101
1,1963-08-01,0.0180,0.0036,0.0100
2,1963-09-01,0.0013,-0.0071,0.0012
3,1963-10-01,-0.0010,0.0280,0.0313
4,1963-11-01,0.0175,-0.0051,-0.0078
